# 5. Evaluation and results

The answers are free text, so exact string matching would fail on any paraphrase. WildGraphBench grades them with an LLM judge (`gpt-5-mini`), using the prompts of its official evaluator:

- **Single-fact and multi-fact:** the judge compares the answer with the reference answer and returns 1 (correct) or 0.
- **Summary:** the judge splits the answer into atomic statements, then measures
  - **recall**: the share of gold statements the answer supports,
  - **precision**: the share of answer statements the gold statements support,
  - **F1**: their harmonic mean.

Judgments are cached, so rerunning this notebook costs nothing unless an answer changed.

In [1]:
from dotenv import load_dotenv

from src import config

load_dotenv(config.PROJECT_ROOT / ".env")
print(f"Domain: {config.DOMAIN} | run mode: {config.RUN_MODE.value}")

Domain: technology | run mode: subset


In [2]:
from src.data import load_run_inputs
from src.question_runner import get_predictions_path
from src.usage_tracking import UsageLedger, get_ledger_path

run_inputs = load_run_inputs(config.DOMAIN, config.RUN_MODE)
ledger = UsageLedger(get_ledger_path(run_inputs.run_directory))
results_directory = config.get_results_directory()

evaluated_systems = [
    system_name
    for system_name in config.SYSTEM_NAMES
    if get_predictions_path(run_inputs.run_directory, system_name).exists()
]
print("Systems with predictions:", evaluated_systems)

Systems with predictions: ['naive_rag', 'lightrag_hybrid', 'graphrag_local', 'graphrag_global']


## 5.1 Judge the answers

In [3]:
from src.evaluation import evaluate_system_predictions

judgments_by_system = {
    system_name: await evaluate_system_predictions(
        system_name,
        questions=run_inputs.questions,
        run_directory=run_inputs.run_directory,
        ledger=ledger,
        max_concurrent_judgments=config.MAX_CONCURRENT_LLM_CALLS,
    )
    for system_name in evaluated_systems
}
judgments_by_system[evaluated_systems[0]].head()

judge naive_rag:   0%|          | 0/9 [00:00<?, ?it/s]

judge lightrag_hybrid:   0%|          | 0/9 [00:00<?, ?it/s]

judge graphrag_local:   0%|          | 0/9 [00:00<?, ?it/s]

judge graphrag_global:   0%|          | 0/9 [00:00<?, ?it/s]

,question_id,question_type,answer_hash,judge_model,correct,reason,recall,precision,f1,answer_statements
0,technology-000,single_fact,e8d14937fe3a2c10,gpt-5-mini,0,Candidate correctly gives daily (33M) and mont...,NaN,NaN,NaN,NaN
1,technology-005,single_fact,125c6d8fee71d5dc,gpt-5-mini,1,Candidate correctly states that the inventory-...,NaN,NaN,NaN,NaN
2,technology-033,single_fact,21f9586ca2b9ecbd,gpt-5-mini,1,The candidate correctly states that a judge (J...,NaN,NaN,NaN,NaN
3,technology-071,multi_fact,eba1c279332504d2,gpt-5-mini,1,Matches reference: correctly names Raw Fury as...,NaN,NaN,NaN,NaN
4,technology-081,multi_fact,9e903542ae9f6200,gpt-5-mini,1,Matches key points: Valve initially planned to...,NaN,NaN,NaN,NaN


## 5.2 The results table

One row per system, question type and metric. Accuracy rows carry a 95% confidence interval (Wilson score method), which stays meaningful on small samples: with 24 summary questions, a single question moves the score by about four points.

In [4]:
from src.evaluation import build_results_table, pivot_results_table

results_table = build_results_table(judgments_by_system, run_directory=run_inputs.run_directory, ledger=ledger)
results_table.to_csv(results_directory / "results.csv", index=False)
print(f"Saved to {results_directory / 'results.csv'}")
results_table.head(8)

Saved to /Users/linafaik/Documents/projects/graph-retrieval-bench/results/technology/subset/results.csv


,system_name,question_type,metric,value,ci_low,ci_high,question_count
0,naive_rag,multi_fact,accuracy,0.666667,0.20766,0.938508,3
1,naive_rag,multi_fact,query_cost_usd_per_question,0.000988,NaN,NaN,3
2,naive_rag,multi_fact,query_tokens_per_question,6248.666667,NaN,NaN,3
3,naive_rag,multi_fact,latency_seconds_per_question,2.303000,NaN,NaN,3
4,naive_rag,single_fact,accuracy,0.666667,0.20766,0.938508,3
5,naive_rag,single_fact,query_cost_usd_per_question,0.000992,NaN,NaN,3
6,naive_rag,single_fact,query_tokens_per_question,6143.000000,NaN,NaN,3
7,naive_rag,single_fact,latency_seconds_per_question,2.647667,NaN,NaN,3


In [5]:
pivot_results_table(results_table).T.style.format("{:,.4f}")

## 5.3 Where does each system win?

In [6]:
from src.visualization import plot_cost_versus_accuracy, plot_indexing_cost_and_time, plot_quality_by_question_type

plot_quality_by_question_type(results_table).show()

The paper's expectation, to check against the chart:

- **Single-fact:** vector retrieval stays competitive, because one well-matched chunk is enough.
- **Multi-fact:** graph methods should lead, because relations connect facts spread over several pages.
- **Summary:** vector retrieval had the best F1 in the paper. Graph summaries filter the noisy evidence too aggressively when breadth matters.

The whiskers show how much of any gap could be sampling noise.

## 5.4 What does the quality cost?

In [7]:
plot_cost_versus_accuracy(results_table).show()

In [8]:
plot_indexing_cost_and_time(results_table).show()

Indexing is paid once; querying is paid for every question. A graph index only pays off when its quality gain matters on the questions actually asked.

## 5.5 Evaluation cost and reproducibility

The judge's own cost is reported separately: it is part of the benchmark, not of any system. The manifest records models, prices, versions and the dataset revision behind the table.

In [9]:
from src.evaluation import write_run_manifest
from src.usage_tracking import Phase

usage_records = ledger.load_records()
judging_cost_usd = usage_records.loc[usage_records["phase"] == Phase.JUDGING.value, "cost_usd"].sum()
print(f"Judging cost: ${judging_cost_usd:.4f}")
print(f"Total spend of this run: ${usage_records['cost_usd'].sum():.4f}")

manifest_path = write_run_manifest(
    results_directory,
    question_count=len(run_inputs.questions),
    document_count=len(run_inputs.documents),
)
print(f"Manifest: {manifest_path}")

Judging cost: $0.0513
Total spend of this run: $0.7420
Manifest: /Users/linafaik/Documents/projects/graph-retrieval-bench/results/technology/subset/run_manifest.json


## Limitations

- **Native prompts.** Each framework answers with its own prompt, so the comparison measures each framework as shipped, not retrieval alone.
- **One domain.** Technology covers a single topic, Steam. The paper averages twelve domains.
- **Judge variance.** `gpt-5-mini` cannot run at temperature 0. The cached judgments keep this table stable, but a fresh judging run may move scores by a few points.
- **Subset runs.** A subset corpus contains almost only relevant pages. Its scores check the pipeline and nothing more.